# AMAGO · Imagen → modelo 3D (TRELLIS, gratis)

**Antes de pulsar Run All**, en el panel de la derecha (*Session options*):
1. **Accelerator → GPU T4 x2** (no uses P100: no es compatible).
2. **Internet → On** (hace falta tener el teléfono verificado en tu cuenta de Kaggle).

Luego pulsa **Run All**. Tarda unos 10-15 min en instalar y 2–4 min por personaje.
Al final aparece el archivo **`amago_modelos.zip`** en *Output* (panel derecho → `/kaggle/working`) para descargarlo.

Las imágenes se cogen solas de GitHub (`amago/art/concepts/` en la rama del proyecto). También puedes añadir imágenes tuyas con *Add Input → Upload*.


In [ ]:
# 1) Comprobar la tarjeta gráfica
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version
!which nvcc || ls /usr/local/cuda*/bin/nvcc

In [ ]:
%%bash
# 2) Instalar TRELLIS (unos 10-15 minutos). Usa el PyTorch que ya trae Kaggle.
set -ex
python -c "import sys, torch; print(sys.version); print('torch', torch.__version__, 'cuda', torch.version.cuda, torch.cuda.is_available())"
export CUDA_HOME=${CUDA_HOME:-/usr/local/cuda}
if [ ! -x "$CUDA_HOME/bin/nvcc" ]; then echo "NO HAY NVCC en $CUDA_HOME — manda captura de esto"; ls /usr/local; exit 1; fi
$CUDA_HOME/bin/nvcc --version | tail -2
CU=$(python -c "import torch; a, b = torch.version.cuda.split('.')[:2]; print(int(a) * 10 + int(b))")
if [ "$CU" -ge 126 ]; then SP=spconv-cu126; elif [ "$CU" -ge 124 ]; then SP=spconv-cu124; elif [ "$CU" -ge 121 ]; then SP=spconv-cu121; else SP=spconv-cu120; fi
pip install -q $SP easydict rembg onnxruntime xatlas pyvista pymeshfix igraph trimesh imageio imageio-ffmpeg ninja
pip install -q git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8
pip install -q git+https://github.com/NVlabs/nvdiffrast.git@v0.3.3
export TORCH_CUDA_ARCH_LIST="7.5"
rm -rf /tmp/mip && git clone -q --depth 1 https://github.com/autonomousvision/mip-splatting.git /tmp/mip
pip install -q --no-build-isolation /tmp/mip/submodules/diff-gaussian-rasterization/
rm -rf /kaggle/working/TRELLIS && git clone -q --depth 1 --recursive https://github.com/microsoft/TRELLIS.git /kaggle/working/TRELLIS
sed -i '/trellis_text_to_3d/d' /kaggle/working/TRELLIS/trellis/pipelines/__init__.py
# imágenes y sustitutos de xformers/kaolin desde GitHub
rm -rf /tmp/repo
git clone -q --depth 1 --filter=blob:none --sparse -b claude/eager-gates-4i58a1 https://github.com/enriquemateo/maker-portfolio.git /tmp/repo
cd /tmp/repo && git sparse-checkout set amago/art/concepts amago/art/kaggle/shim
ls -la /tmp/repo/amago/art/concepts
python -c "import spconv, diff_gaussian_rasterization; import nvdiffrast.torch as dr; dr.RasterizeCudaContext(); print('INSTALACION OK')"


In [ ]:
# 4) Generar los modelos 3D
import os, sys, glob, zipfile, traceback, gc
os.environ['ATTN_BACKEND'] = 'xformers'
os.environ['SPCONV_ALGO'] = 'native'
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'
sys.path.insert(0, '/kaggle/working/TRELLIS')
sys.path.insert(0, '/tmp/repo/amago/art/kaggle/shim')
import numpy as np, torch, imageio
from PIL import Image
from IPython.display import display
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import render_utils, postprocessing_utils

SEEDS = [1, 2]          # dos versiones de cada personaje para elegir la mejor
OUT = '/kaggle/working/modelos'
os.makedirs(OUT, exist_ok=True)

imgs = sorted(glob.glob('/tmp/repo/amago/art/concepts/*.png') + glob.glob('/tmp/repo/amago/art/concepts/*.jp*g') +
              glob.glob('/kaggle/input/**/*.png', recursive=True) + glob.glob('/kaggle/input/**/*.jp*g', recursive=True))
print(len(imgs), 'imágenes:', [os.path.basename(p) for p in imgs])

pipe = TrellisImageTo3DPipeline.from_pretrained('microsoft/TRELLIS-image-large')
pipe.cuda()

for path in imgs:
    name = os.path.splitext(os.path.basename(path))[0]
    for seed in SEEDS:
        tag = f'{name}_s{seed}'
        if os.path.exists(f'{OUT}/{tag}.glb'):
            continue
        try:
            print('>>', tag)
            out = pipe.run(Image.open(path), seed=seed, formats=['mesh', 'gaussian'])
            frames = render_utils.render_video(out['gaussian'][0], num_frames=4, bg_color=(1, 1, 1), verbose=False)['color']
            prev = np.concatenate(frames, axis=1)
            imageio.imwrite(f'{OUT}/{tag}_preview.png', prev)
            display(Image.fromarray(prev).resize((1024, 256)))
            glb = postprocessing_utils.to_glb(out['gaussian'][0], out['mesh'][0], simplify=0.9, texture_size=2048, verbose=False)
            glb.export(f'{OUT}/{tag}.glb')
            print('   OK', tag, round(os.path.getsize(f'{OUT}/{tag}.glb') / 1e6, 1), 'MB')
        except Exception:
            traceback.print_exc()
        finally:
            out = None; gc.collect(); torch.cuda.empty_cache()

with zipfile.ZipFile('/kaggle/working/amago_modelos.zip', 'w') as z:
    for f in sorted(glob.glob(f'{OUT}/*')):
        z.write(f, os.path.basename(f))
print('LISTO → descarga /kaggle/working/amago_modelos.zip')